# 04 — Regression on ADMET Properties

End-to-end walkthrough: train CAGEFusion to predict **continuous** ADMET
properties from molecular structure.

**What you'll learn**
- `model_task="regression"` in `CageFusionConfig`
- `AutoCageFusion.from_config` → dispatches to `CAGEFusionForRegression` (MSE loss, no sigmoid)
- Training metrics: RMSE / MAE / R² per task (computed automatically by the trainer)
- Held-out test evaluation and inference via `AutoCageFusion.from_pretrained`

**Key differences from classification**

| | Classification | Regression |
|---|---|---|
| `model_task` | `"classification"` | `"regression"` |
| Model head | sigmoid + BCEWithLogitsLoss | linear + MSELoss |
| Training metrics | AUC / MCC / PR-AUC | RMSE / MAE / R² |
| Best checkpoint | highest val AUC | lowest val RMSE |
| Inference | `CageFusionPipeline` | `AutoCageFusion.from_pretrained` |
| Label values | 0 / 1 | any float |


In [1]:
%matplotlib inline

import os
import pandas as pd
import numpy as np
import torch

from cage_fusion import CageFusionConfig, AutoCageFusion
from cage_fusion.data import CageFusionDataModule
from cage_fusion.training import Trainer, TrainingArguments
from cage_fusion.utils.device_utils import move_bmg_to_device

## 1. Prepare your CSV

Your CSV needs:
- A `SMILES` column
- One or more **float** label columns (any continuous values — no 0/1 restriction)

Below we create a toy ADMET dataset with four representative endpoints:

| Property | Unit | Typical range |
|---|---|---|
| `logP` | log₁₀ (octanol/water) | −2 … 5 |
| `aqueous_logS` | log₁₀ (mol/L) | −6 … 0 |
| `Caco2_perm` | log₁₀ (cm/s) | −7 … −4 |
| `hERG_pIC50` | −log₁₀ (IC₅₀/M) | 3 … 7 |

Replace with your real dataset — the rest of the notebook requires no changes.

In [2]:
os.makedirs("data", exist_ok=True)

# Toy dataset — molecules with approximate experimental values
admet_data = [
    # SMILES                                             logP   logS    Caco2   hERG_pIC50
    {"SMILES": "CC(=O)Oc1ccccc1C(=O)O",                 "logP":  1.19, "aqueous_logS": -2.26, "Caco2_perm": -5.15, "hERG_pIC50": 3.8},  # aspirin
    {"SMILES": "c1ccc2ccccc2c1",                          "logP":  3.37, "aqueous_logS": -3.30, "Caco2_perm": -4.85, "hERG_pIC50": 4.1},  # naphthalene
    {"SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",           "logP": -0.07, "aqueous_logS": -1.38, "Caco2_perm": -5.70, "hERG_pIC50": 3.2},  # caffeine
    {"SMILES": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",    "logP":  3.10, "aqueous_logS": -4.80, "Caco2_perm": -5.20, "hERG_pIC50": 4.9},  # testosterone
    {"SMILES": "O=C(O)c1ccccc1O",                         "logP":  2.26, "aqueous_logS": -2.10, "Caco2_perm": -5.40, "hERG_pIC50": 3.5},  # salicylic acid
    {"SMILES": "C1CCCCC1",                                "logP":  3.44, "aqueous_logS": -3.44, "Caco2_perm": -4.60, "hERG_pIC50": 3.0},  # cyclohexane
    {"SMILES": "CCO",                                     "logP": -0.31, "aqueous_logS":  0.00, "Caco2_perm": -6.50, "hERG_pIC50": 2.9},  # ethanol
    {"SMILES": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",             "logP":  3.97, "aqueous_logS": -3.97, "Caco2_perm": -4.80, "hERG_pIC50": 4.5},  # ibuprofen
    {"SMILES": "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3", "logP": 3.31, "aqueous_logS": -4.40, "Caco2_perm": -4.50, "hERG_pIC50": 5.8},  # crystal violet
    {"SMILES": "O=C1c2ccccc2C(=O)c3ccccc13",             "logP":  3.43, "aqueous_logS": -5.00, "Caco2_perm": -5.10, "hERG_pIC50": 4.2},  # anthraquinone
    {"SMILES": "c1ccc(cc1)N",                             "logP":  0.90, "aqueous_logS": -1.90, "Caco2_perm": -5.80, "hERG_pIC50": 3.6},  # aniline
    {"SMILES": "CC(=O)N",                                 "logP": -1.26, "aqueous_logS":  0.55, "Caco2_perm": -6.80, "hERG_pIC50": 2.5},  # acetamide
    {"SMILES": "OC(=O)CC(O)(CC(=O)O)C(=O)O",             "logP": -1.72, "aqueous_logS":  0.20, "Caco2_perm": -6.90, "hERG_pIC50": 2.8},  # citric acid
    {"SMILES": "CCCCCCCC(=O)O",                           "logP":  3.05, "aqueous_logS": -3.05, "Caco2_perm": -4.90, "hERG_pIC50": 3.3},  # octanoic acid
    {"SMILES": "c1ccc(cc1)C(=O)O",                        "logP":  1.87, "aqueous_logS": -2.10, "Caco2_perm": -5.25, "hERG_pIC50": 3.7},  # benzoic acid
    {"SMILES": "CCOC(=O)c1ccc(cc1)N",                    "logP":  1.96, "aqueous_logS": -2.60, "Caco2_perm": -5.35, "hERG_pIC50": 3.9},  # ethyl 4-aminobenzoate
    {"SMILES": "CC(C)(C)c1ccc(cc1)O",                    "logP":  3.31, "aqueous_logS": -3.80, "Caco2_perm": -4.70, "hERG_pIC50": 4.3},  # 4-tert-butylphenol
    {"SMILES": "OC1=CC=C(C=C1)O",                        "logP":  0.59, "aqueous_logS": -1.60, "Caco2_perm": -5.90, "hERG_pIC50": 3.4},  # hydroquinone
]

df = pd.DataFrame(admet_data)
df.to_csv("data/admet_toy.csv", index=False)

print(f"Dataset: {len(df)} rows")
print(df.describe())

Dataset: 18 rows
            logP  aqueous_logS  Caco2_perm  hERG_pIC50
count  18.000000     18.000000   18.000000   18.000000
mean    1.799444     -2.497222   -5.411111    3.744444
std     1.764424      1.653541    0.727113    0.811840
min    -1.720000     -5.000000   -6.900000    2.500000
25%     0.667500     -3.710000   -5.775000    3.225000
50%     2.110000     -2.430000   -5.225000    3.650000
75%     3.310000     -1.675000   -4.862500    4.175000
max     3.970000      0.550000   -4.500000    5.800000


## 2. Build the data module

`from_csv` works identically for regression — labels are stored as float32.
No binary requirement.

In [3]:
LABEL_COLS      = ["logP", "aqueous_logS", "Caco2_perm", "hERG_pIC50"]
CHECKPOINT_DIR  = "data/tmp/cage_fusion_admet"

dm = CageFusionDataModule.from_csv(
    csv_path="data/admet_toy.csv",
    label_cols=LABEL_COLS,
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
    val_split=0.15,
    test_split=0.10,
    cache_dir="data/tmp/admet_features",
    batch_size=8,
)

print("Label names :", dm.label_names)
print("Train batches:", len(dm.train_loader))
print("Val batches  :", len(dm.val_loader))
print("Test batches :", len(dm.test_loader) if dm.test_loader else "None")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

                  Featurisation parameters                   
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter  ┃ Value                                        ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ HDF5 path  │ data/tmp/admet_features/train_cage_fusion.h5 │
│ N samples  │ 13                                           │
│ seq_len    │ 512                                          │
│ embed_dim  │ 384                                          │
│ aux_dim    │ 217                                          │
│ num_labels │ 4                                            │
│ ids        │ False                                        │
│ batch_size │ 32                                           │
└────────────┴──────────────────────────────────────────────┘

INFO     Initialised HDF5 at data/tmp/admet_features/train_cage_fusion.h5 | N=13 emb=float32 ids=int32 aux_dim=217 labels=4


Featurising train: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.62it/s]

INFO     Normalising auxiliary features in data/tmp/admet_features/train_cage_fusion.h5 ...


...

INFO     Initialised HDF5 at data/tmp/admet_features/test_cage_fusion.h5 | N=2 emb=float32 ids=int32 aux_dim=217 labels=4


Featurising test: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.10it/s]

INFO     Normalising auxiliary features in data/tmp/admet_features/test_cage_fusion.h5 ...



Normalising test: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2043.01it/s]

INFO     Wrote 'auxiliary_features_normalized' to data/tmp/admet_features/test_cage_fusion.h5
INFO     [wk0/1] Dataset ready: N=2 graph_cache=objects emb_shape=(2, 512, 384)
Label names : ['logP', 'aqueous_logS', 'Caco2_perm', 'hERG_pIC50']
Train batches: 2
Val batches  : 1
Test batches : 1


## 3. Configure the model

The only required change from classification is `model_task="regression"`.
`AutoCageFusion.from_config` then dispatches to `CAGEFusionForRegression`,
which uses a **linear output head** and **MSELoss** instead of sigmoid + BCE.

In [4]:
config = CageFusionConfig(
    num_labels=len(dm.label_names),
    model_task="regression",
    label_names=dm.label_names,
    attn_mode="cross",
    use_fg_prompt=True,
    hidden_size=128,
)

print(config)

CageFusionConfig(num_labels=4, model_task='regression', attn_mode='cross', hidden_size=128, fusion_dim=901)


## 4. Build the model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = AutoCageFusion.from_config(config).to(device)

print(type(model).__name__)   # CAGEFusionForRegression
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

INFO     CAGEFusionModel initialised (attn_mode=cross)
CAGEFusionForRegression
Total parameters    : 7,351,316
Trainable parameters: 7,351,316


## 5. Train

The trainer automatically detects `model_task="regression"` and uses
RMSE / MAE / R² metrics throughout — no sigmoid is applied, no AUC/MCC noise.
Per-epoch tables show RMSE / MAE / R² per task; `best_model.pt` is
selected by best val RMSE (lower is better).


In [6]:
args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    checkpoints_dir=CHECKPOINT_DIR,
    num_epochs=5,
    batch_size=8,
    learning_rate=3e-4,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

history = trainer.train()
print(f"Training complete.  Best val RMSE: {min(history['val_rmse']):.4f}")

INFO     Auto-built Adam optimizer  lr=3.00e-04  wd=0.00e+00  params=90
INFO     Training from epoch 1 to 5 | train batches: 2 | val batches: 1
INFO     Trainable params: 7,351,316


──────────────────────────────────────────────────── Epoch 1/5 ────────────────────────────────────────────────────

Train epoch 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  7.90it/s]

INFO     Top-5 attended FGs:
  1. pyrimidine                avg=1.0000
  2. amide                     avg=1.0000
  3. aniline                   avg=0.5126
  4. hydroxyl                  avg=0.4710
  5. carbonyl                  avg=0.3939
INFO     Epoch train | loss=10.2693 rmse=3.1961 mae=2.9150 r2=-15.6654



Evaluate: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 66.80it/s]

...

Train Loss: 5.1139 (-0.1487)

       Validation Metrics        
┏━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric ┃    Value ┃         Δ ┃
┡━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ Loss   │   5.1559 │ (-0.1975) │
│ RMSE   │   2.2444 │ (-0.0453) │
│ MAE    │   2.1131 │ (-0.0471) │
│ R²     │ -10.9502 │ (+0.7715) │
└────────┴──────────┴───────────┘

                      Per-Task Validation Metrics                       
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Task         ┃            RMSE ┃             MAE ┃                R² ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ logP         │ 2.214 (+0.0339) │ 2.110 (+0.0380) │  -9.950 (-0.3328) │
│ aqueous_logS │ 2.223 (-0.0513) │ 2.098 (-0.0537) │  -8.138 (+0.4263) │
│ Caco2_perm   │ 1.785 (-0.1082) │ 1.744 (-0.1097) │ -21.138 (+2.7645) │
│ hERG_pIC50   │ 2.756 (-0.0558) │ 2.499 (-0.0632) │  -4.574 (+0.2280) │
├──────────────┼─────────────────┼─────────────────┼───────────────────┤
│ Macro-Avg    │          2.2444 │          2.1131 │          -10.9502 │
└──────────────┴─────────────────┴─────────────────┴───────────────────┘

                     Learned Modality Scalers                      
┏━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Scaler      ┃   Value ┃ Avg. Rep Norm ┃ Scaled Norm ┃ Δ (Value) ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ scale_graph │ 10.0000 │        2.3691 │     23.6908 │ (-0.0002) │
│ scale_attn  │  1.0003 │       19.0742 │     19.0792 │ (+0.0001) │
│ scale_aux   │  0.4990 │       21.3159 │     10.6365 │ (+0.0003) │
└─────────────┴─────────┴───────────────┴─────────────┴───────────┘

INFO     Saved model to data/tmp/cage_fusion_admet
INFO     New best RMSE=2.2444 → data/tmp/cage_fusion_admet/best_model.pt
INFO     New best MAE=2.1131 → data/tmp/cage_fusion_admet/best_model_mcc.pt
INFO     Training complete.
INFO     Saved training history to data/tmp/cage_fusion_admet/training_history.csv
Training complete.  Best val RMSE: 2.2444


## 6. Plot training loss

In [7]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

plot_cfg = [
    ("Loss (MSE)",  "train_loss",  "val_loss"),
    ("RMSE",        "train_rmse",  "val_rmse"),
    ("MAE",         "train_mae",   "val_mae"),
    ("R²",          "train_r2",    "val_r2"),
]

for ax, (title, train_key, val_key) in zip(axes, plot_cfg):
    ax.plot(epochs, history[train_key], label="Train")
    ax.plot(epochs, history[val_key],   label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True)

plt.suptitle("Training History — Regression", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Held-out test set evaluation

The trainer monitors the validation set and saves `best_model.pt` by RMSE.
Run the saved best model on the **test set** for an unbiased performance estimate.


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the best checkpoint (selected by lowest val RMSE during training)
best_model = AutoCageFusion.from_pretrained(CHECKPOINT_DIR).to(device)
best_model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in dm.test_loader:
        bmg, token_embs, attn_mask, aux_feats, labels, input_ids, smiles_batch, _, _ = batch
        bmg        = move_bmg_to_device(bmg, device)
        token_embs = token_embs.to(device)
        attn_mask  = attn_mask.to(device)
        aux_feats  = aux_feats.to(device)
        input_ids  = input_ids.to(device)

        out = best_model(
            bmg=bmg,
            sequence_embeddings=token_embs,
            attn_mask=attn_mask,
            aux_feats=aux_feats,
            input_ids_batch=input_ids,
            smiles_batch=smiles_batch,
        )
        all_preds.append(out.logits.cpu().numpy())
        all_labels.append(labels.numpy())

preds  = np.vstack(all_preds)
labels_np = np.vstack(all_labels)

print(f"{'Property':<18} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 46)
for i, name in enumerate(dm.label_names):
    rmse = mean_squared_error(labels_np[:, i], preds[:, i]) ** 0.5
    mae  = mean_absolute_error(labels_np[:, i], preds[:, i])
    r2   = r2_score(labels_np[:, i], preds[:, i]) if len(labels_np) > 1 else float("nan")
    print(f"{name:<18} {rmse:>8.3f} {mae:>8.3f} {r2:>8.3f}")

INFO     CAGEFusionModel initialised (attn_mode=cross)
Property               RMSE      MAE       R²
----------------------------------------------
logP                  1.763    1.445   -1.616
aqueous_logS          1.456    1.375   -6.844
Caco2_perm            1.953    1.950 -168.511
hERG_pIC50            2.214    2.209 -216.934


> **Toy dataset caveat:** The test set has only 1–2 molecules, so
> RMSE / MAE / R² are not statistically meaningful here.  With a real
> ADMET dataset (hundreds–thousands of compounds) these metrics are reliable.

## 8. Save checkpoint, scaler, and config

In [9]:
dm.save_scaler(CHECKPOINT_DIR)
config.save_pretrained(CHECKPOINT_DIR)

print("Files saved:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    print(" ", f)

Files saved:
  aux_features_scaler.pkl
  best_model.pt
  best_model_mcc.pt
  config.json
  latest_checkpoint.pt
  loss_curve.png
  mae_curve.png
  pytorch_model.bin
  r2_curve.png
  rmse_curve.png
  training_history.csv


## 9. Inference

`CageFusionPipeline` is classification-only (applies sigmoid + thresholding).
For regression, load via `AutoCageFusion.from_pretrained`, which reads
`config.json` and dispatches to `CAGEFusionForRegression` automatically.

In [10]:
# Load the best regression model
loaded_model = AutoCageFusion.from_pretrained(CHECKPOINT_DIR).to(device)
loaded_model.eval()
print(type(loaded_model).__name__)   # CAGEFusionForRegression

# Save new SMILES to a temporary CSV
new_smiles = [
    "CC(=O)Oc1ccccc1C(=O)O",   # aspirin
    "c1ccc2ccccc2c1",           # naphthalene
    "CCO",                      # ethanol
    "c1ccc(cc1)N",              # aniline
]
os.makedirs("data/tmp", exist_ok=True)
pd.DataFrame({"SMILES": new_smiles}).to_csv("data/tmp/new_compounds.csv", index=False)

# Featurise — pass the scaler fitted during training so descriptors are on the same scale
dm_infer = CageFusionDataModule.for_inference(
    csv_path="data/tmp/new_compounds.csv",
    label_cols=[],
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
    scaler=dm.scaler,
    cache_dir="data/tmp/admet_infer_features",
    batch_size=8,
)

results = []
with torch.no_grad():
    for batch in dm_infer.test_loader:
        bmg, token_embs, attn_mask, aux_feats, _, input_ids, smiles_batch, _, _ = batch
        bmg        = move_bmg_to_device(bmg, device)
        token_embs = token_embs.to(device)
        attn_mask  = attn_mask.to(device)
        aux_feats  = aux_feats.to(device)
        input_ids  = input_ids.to(device)

        out = loaded_model(
            bmg=bmg,
            sequence_embeddings=token_embs,
            attn_mask=attn_mask,
            aux_feats=aux_feats,
            input_ids_batch=input_ids,
            smiles_batch=smiles_batch,
        )
        for smi, pred_row in zip(smiles_batch, out.logits.cpu().numpy()):
            row = {"SMILES": smi}
            row.update(dict(zip(dm.label_names, pred_row.tolist())))
            results.append(row)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

INFO     CAGEFusionModel initialised (attn_mode=cross)
CAGEFusionForRegression


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

                     Featurisation parameters                      
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter  ┃ Value                                              ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ HDF5 path  │ data/tmp/admet_infer_features/infer_cage_fusion.h5 │
│ N samples  │ 4                                                  │
│ seq_len    │ 512                                                │
│ embed_dim  │ 384                                                │
│ aux_dim    │ 217                                                │
│ num_labels │ 0                                                  │
│ ids        │ False                                              │
│ batch_size │ 32                                                 │
└────────────┴────────────────────────────────────────────────────┘

INFO     Initialised HDF5 at data/tmp/admet_infer_features/infer_cage_fusion.h5 | N=4 emb=float32 ids=int32 aux_dim=217 labels=0


Featurising infer: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.13it/s]

INFO     Normalising auxiliary features in data/tmp/admet_infer_features/infer_cage_fusion.h5 ...



Normalising infer: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2075.36it/s]

INFO     Wrote 'auxiliary_features_normalized' to data/tmp/admet_infer_features/infer_cage_fusion.h5
INFO     [wk0/1] Dataset ready: N=4 graph_cache=objects emb_shape=(4, 512, 384)


               SMILES     logP  aqueous_logS  Caco2_perm  hERG_pIC50
CC(=O)Oc1ccccc1C(=O)O 0.755325     -1.365747   -3.085421    1.741009
       c1ccc2ccccc2c1 0.914936     -1.444617   -3.015462    1.740428
                  CCO 0.616281     -1.312844   -3.147572    1.698623
          c1ccc(cc1)N 0.806532     -1.386271   -3.062247    1.740829


## 10. Push to HuggingFace Hub (optional)

Same as classification — specify `model=` to select which checkpoint to upload.

When reloading, `AutoCageFusion.from_pretrained` reads `config.json` and
automatically reconstructs `CAGEFusionForRegression`.

In [ ]:
from cage_fusion import CageFusionPipeline

HF_REPO_ID = "your-username/cage-fusion-admet"   # ← edit this
HF_TOKEN   = None   # or "hf_..."

# url = CageFusionPipeline.push_to_hub(
#     CHECKPOINT_DIR,
#     repo_id=HF_REPO_ID,
#     model="best",
#     token=HF_TOKEN,
#     private=True,
# )
# print("Uploaded to:", url)

# --- reload from the Hub -----------------------------------------------
# model_hf = AutoCageFusion.from_pretrained(HF_REPO_ID)
# type(model_hf).__name__  -> "CAGEFusionForRegression"